# Lab 3 · Evaluation — Kaggle T4

## What we're doing

Stop saying *"the fine-tune feels better"* and **put numbers on it**. We score the
Lab 2 base model and the fine-tuned adapter on the **same held-out test set**
(the order → JSON task), then run a standard academic **benchmark** for contrast.
The point: a fine-tune can lift *your* task metric a lot while barely moving a
general benchmark — and that's fine. Trust the held-out task eval for your job.

## What you'll see

- **Part A — task eval.** A base-vs-fine-tuned table on three metrics:
  - **json_valid_%** — did it emit parseable JSON at all? (no prose/fences)
  - **exact_match_%** — does the JSON exactly equal the gold object?
  - **&lt;field&gt;_acc_%** — per-key accuracy for intent / item / qty / city
  Expect **exact-match** and the field accuracies to jump after fine-tuning.
- **Part B — benchmark.** `arc_easy` via lm-eval-harness — a general-ability
  score that the fine-tune shouldn't change much.

---

**Run from the Kaggle editor on a T4** (the API push can't request a T4). Then
**Save Version → Save & Run All**.

**Depends on Lab 2.** Reads the adapter + test set that `lab2_finetune_kaggle`
wrote to `/kaggle/working`. Add it as an input:

> Notebook → **Add Input** → **Notebook Output** → pick `aibase-lab2-finetune`.

**Kaggle gotchas:** 'GPU T4 x2' = two T4s → pin to one; no bf16 → **fp16**; torch
pinned to the preinstalled build.

## Cell 1 — single GPU + deps + locate the Lab 2 outputs

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"   # 'GPU T4 x2' = two T4s; pin to one (set BEFORE torch)

import torch
open("/tmp/constraints.txt", "w").write("torch==" + torch.__version__.split("+")[0] + "\n")
!pip -q install -c /tmp/constraints.txt peft lm-eval

import glob
print("torch:", torch.__version__, "| capability sm_%d%d" % torch.cuda.get_device_capability())

# Lab 2's kernel output mounts at /kaggle/input/<slug>/ ; find the adapter + test set.
cands = glob.glob("/kaggle/input/*/adapters/lora") + ["adapters/lora"]
ADAPTER = next((p for p in cands if os.path.isdir(p)), None)
tcands = glob.glob("/kaggle/input/*/data/test.jsonl") + ["data/test.jsonl"]
TEST = next((p for p in tcands if os.path.isfile(p)), None)
assert ADAPTER and TEST, (
    "Missing Lab 2 outputs. Add the 'aibase-lab2-finetune' notebook output as an "
    "input (Add Input -> Notebook Output).")
print("adapter:", ADAPTER, "\ntest:   ", TEST)

## Part A — held-out task eval (base vs fine-tuned)

For every test example we feed the **system + user** turns to each model, parse
the reply, and compare to the gold JSON. Three metrics: **JSON-valid %**,
**exact-match %**, and per-field accuracy (intent / item / qty / city).

In [ ]:
import json, gc
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

BASE = "Qwen/Qwen2.5-3B-Instruct"
FIELDS = ["intent", "item", "qty", "city"]
tests = [json.loads(l) for l in open(TEST)]
print(f"base: {BASE}  test set: {len(tests)} examples")

def gen(tok, model, messages):
    enc = tok.apply_chat_template(messages, add_generation_prompt=True,
                                  return_tensors="pt", return_dict=True)
    enc = {k: v.to(model.device) for k, v in enc.items()}
    with torch.no_grad():
        out = model.generate(**enc, max_new_tokens=128, do_sample=False)
    return tok.decode(out[0][enc["input_ids"].shape[1]:], skip_special_tokens=True).strip()

def try_parse(text):
    t = text.strip().removeprefix("```json").removeprefix("```").removesuffix("```").strip()
    try: return json.loads(t)
    except Exception: return None

def score(tok, model):
    valid = exact = 0; hits = {f: 0 for f in FIELDS}
    for t in tests:
        gold = json.loads(t["messages"][2]["content"])
        pred = try_parse(gen(tok, model, t["messages"][:2]))
        if pred is None: continue
        valid += 1; exact += (pred == gold)
        for f in FIELDS:
            if pred.get(f) == gold.get(f): hits[f] += 1
    n = len(tests)
    return {"json_valid_%": 100*valid/n, "exact_match_%": 100*exact/n,
            **{f"{f}_acc_%": 100*hits[f]/n for f in FIELDS}}

tok = AutoTokenizer.from_pretrained(BASE)
base = AutoModelForCausalLM.from_pretrained(BASE, dtype=torch.float16, device_map="auto").eval()  # fp16
base_m = score(tok, base); del base; gc.collect(); torch.cuda.empty_cache()
ft = PeftModel.from_pretrained(
        AutoModelForCausalLM.from_pretrained(BASE, dtype=torch.float16, device_map="auto"), ADAPTER).eval()
ft_m = score(tok, ft)

print(f"\n{'metric':<16}{'BASE':>10}{'FINE-TUNED':>14}{'Δ':>10}")
print("-" * 50)
for k in base_m:
    print(f"{k:<16}{base_m[k]:>9.1f}%{ft_m[k]:>13.1f}%{ft_m[k]-base_m[k]:>+9.1f}")

## Part B — standardized benchmark (lm-eval-harness)

`arc_easy`, capped at 200 examples for speed. A leaderboard number measures
*general* ability; a fine-tune can lift the task metric above while barely moving
this. Slow on the free GPU — skip if you're short on quota.

In [ ]:
del ft; gc.collect(); torch.cuda.empty_cache()
!lm_eval --model hf \
  --model_args pretrained=Qwen/Qwen2.5-3B-Instruct,dtype=float16 \
  --tasks arc_easy --device cuda:0 --batch_size auto --limit 200

## Teaching points

- **Contamination:** a high benchmark score can be memorized test data. Your own
  held-out task eval (Part A) is harder to game — trust it more.
- **One number lies:** report quality **and** speed **and** cost together (tie
  back to Lab 1's tokens/s and VRAM).
- **Eval gates CI:** in MLOps this script becomes a pipeline gate — a regression
  on `exact_match_%` blocks the deploy.
- **Part C (optional):** re-run Part A against a quantized serve (the Lab 1 Ollama
  Q4 endpoint) to measure whether Q4 costs task accuracy. *Measure, don't assume.*